# 1.3 Demo - Automating SCD Type 2 with AUTO CDC in Spark Declarative Pipelines
**Lesson:** Change Data Capture (CDC) Review

A CDC feed delivers **INSERT / UPDATE / DELETE events** from an operational database. Applying them by hand with `MERGE` is easy to get wrong: events arrive **out of order**, one batch holds several changes for the same key, and deletes must not be undone by late updates.

`AUTO CDC INTO` (Python: `dp.create_auto_cdc_flow`, formerly `APPLY CHANGES INTO`) handles all of that declaratively:

| Parameter | Meaning |
|---|---|
| `keys` | Columns that identify a record |
| `sequence_by` | Ordering column. The highest sequence wins, regardless of arrival order |
| `apply_as_deletes` | Expression that marks an event as a delete |
| `except_column_list` / `column_list` | Which columns land in the target |
| `stored_as_scd_type` | `1` = current state only, `2` = full history with `__START_AT` / `__END_AT` |
| `track_history_column_list` / `track_history_except_column_list` | SCD2: which column changes open a new version |

The demo replays **three scripted CDC batches** and shows exactly what each target looks like after each one.

## A. Classroom setup

In [ ]:
from adelab.classroom import setup
from adelab import datagen

DA = setup("1.3", reset=True)
CDC_SOURCE = f"{DA.paths.landing}/customers_cdc"

## B. The pipeline
`customers_cdc_bronze` → `customers_cdc_clean` (expectations) → AUTO CDC into **two** targets: `customers_scd1` and `customers_scd2`, plus a `customers_current` view. For SCD2, an email change alone does **not** open a new version (`track_history_except_column_list=["email"]`). It updates the current row in place.

![CDC events for customers 2 and 3 arrive in three batches, including a delete and two late events; AUTO CDC orders them by sequence_num and writes customers_scd1 with only the current Busan row and customers_scd2 with versions bounded by __START_AT and __END_AT](../images/1.3-scd1-vs-scd2.svg)

**How to read it:** the left table is what the three batches below will land (customers 2 and 3 only). The right side is the end state after batch 2: `customers_scd1` keeps one row per live key, `customers_scd2` keeps every version. Shaded rows are the late events and the versions they produce.

In [ ]:
DA.show_source("pipelines/auto_cdc")

In [ ]:
pipe = DA.pipeline("auto_cdc", source="pipelines/auto_cdc", configuration={"cdc_source": CDC_SOURCE})

## C. Batch 0: initial load (4 inserts)

In [ ]:
datagen.land_customer_cdc(CDC_SOURCE, 0)
display(spark.read.json(f"{CDC_SOURCE}/cdc-000.json"))

In [ ]:
update = pipe.run()

In [ ]:
%%sql
SELECT * FROM customers_scd2 ORDER BY customer_id, __START_AT

## D. Batch 1: updates, a delete, and two changes for one key
- customer 1 moves Lisbon → Porto (seq 2)
- customer 2 changes **only email** (seq 2), then moves to Busan (seq 4). Both events are in the **same batch**
- customer 3 is **deleted** (seq 3)
- customer 5 is new

In [ ]:
datagen.land_customer_cdc(CDC_SOURCE, 1)
display(spark.read.json(f"{CDC_SOURCE}/cdc-001.json").orderBy("customer_id", "sequence_num"))

In [ ]:
update = pipe.run()

In [ ]:
%%sql
SELECT * FROM customers_scd2 ORDER BY customer_id, __START_AT

Check the result:
- Customer 2's email change at seq 2 updated version 1 **in place** (email is not tracked). The move at seq 4 opened a new version.
- Customer 3's only version was **closed** at seq 3 (`__END_AT = 3`). No open version means the customer is deleted.

## E. Batch 2: late, out-of-order events
- customer 2: an event with **seq 3** (Incheon) arrives **after** seq 4 was already applied
- customer 3: an update with **seq 2** arrives **after** the delete at seq 3
- customer 4: a normal update
- customer 1: a duplicate of the current values (no tracked column changes)

In [ ]:
datagen.land_customer_cdc(CDC_SOURCE, 2)
display(spark.read.json(f"{CDC_SOURCE}/cdc-002.json").orderBy("customer_id"))

In [ ]:
update = pipe.run()

In [ ]:
%%sql
SELECT * FROM customers_scd2 ORDER BY customer_id, __START_AT

![SCD2 timeline along sequence_num: customer 2 after batch 1 has Seoul then Busan; after batch 2 the late Incheon version is spliced in between seq 3 and 4; customer 3 has Osaka, then the late Kyoto version, closed by the delete at seq 3](../images/1.3-scd2-timeline.svg)

**How to read it:** each bar is one row of `customers_scd2`, from `__START_AT` to `__END_AT`. Compare the first two lanes with the query results before and after batch 2. The DELETE marker at seq 3 closes Kyoto: no bar continues past it, so customer 3 is not in `customers_current`.

AUTO CDC rebuilt history in **sequence order**, not arrival order:
- Customer 2 now has Seoul [1,3) → **Incheon [3,4)** → Busan [4, open). The late version was inserted in the middle of the history.
- Customer 3 has Osaka [1,2) → Kyoto [2,3), and is still **deleted** at 3.
- Customer 1 got no new version, because no tracked column changed.

SCD1 keeps only the latest state. The late Incheon event (seq 3 < 4) and the late update to deleted customer 3 are **ignored**:

In [ ]:
%%sql
SELECT * FROM customers_scd1 ORDER BY customer_id

In [ ]:
%%sql
SELECT * FROM customers_current ORDER BY customer_id

## F. Point-in-time queries on SCD2
"What did customer data look like as of sequence 3?" A version is valid for `__START_AT <= t < __END_AT`.

In [ ]:
%%sql
SELECT customer_id, name, city
FROM customers_scd2
WHERE __START_AT <= 3 AND (__END_AT IS NULL OR __END_AT > 3)
ORDER BY customer_id

## G. For comparison: the hand-written MERGE
Without AUTO CDC you would write something like this for **SCD1** alone. It still breaks on out-of-order data unless you also store and compare sequence numbers, keep tombstones for deletes, and deduplicate within each batch:
```sql
MERGE INTO customers t
USING (SELECT * FROM updates QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY sequence_num DESC) = 1) s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED AND s.operation != 'DELETE' THEN INSERT *
```
SCD2 by hand is much longer: close the old version, insert the new one, and handle late arrivals that must be spliced into the middle of the history.

**SQL syntax on Databricks** (see `databricks/sql/auto_cdc.sql`):
```sql
CREATE OR REFRESH STREAMING TABLE customers_scd2;
CREATE FLOW customers_scd2_flow AS AUTO CDC INTO customers_scd2
FROM STREAM(customers_cdc_clean)
KEYS (customer_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num)
STORED AS SCD TYPE 2
TRACK HISTORY ON * EXCEPT (email);
```

In [ ]:
DA.cleanup()